## API Query - Conversion to Points - With pagination
Pagination in API results is a technique used to manage and retrieve large sets of data efficiently by breaking them into smaller, more manageable chunks (AKA pages). In the breweries API, the number of results per page and the page number can be specified as URL parameters. We can use these in a loop to get all the data.

### 1. Import libraries and define the API endpoint
- Note the per_page and page parameters. More info here: https://openbrewerydb.org/documentation

In [ ]:
import requests
import pandas as pd
import arcpy

# Define the API endpoint
url = 'https://api.openbrewerydb.org/v1/breweries?by_country=united%20states&by_state=wisconsin&per_page=20&page=1'

### 2. Get the data. Use a while loop and a counter variable (page_num) to go through the pages until the end, progressively appending data to a pandas DataFrame.

In [ ]:
# Send a GET request to the API
response = requests.get(url)

df = pd.DataFrame()
page_num = 1

# Check if the request was successful
if response.status_code == 200:
    # Parse the JSON response
    data = response.json()

    while len(data) > 0:
        print('Page: {} Number of results on this page: {}'.format(page_num, len(data)))
        # Convert the JSON data to a pandas DataFrame
        df = pd.concat([df, pd.DataFrame(data)])
        url = 'https://api.openbrewerydb.org/v1/breweries?by_country=united%20states&by_state=wisconsin&per_page=20&page=' + str(page_num)
        page_num += 1
        response = requests.get(url)
        data = response.json()
        
else:
    print(f"Failed to retrieve data: {response.status_code}")

### 3. Save results as CSV, use XY Table to Point to bring data into GIS.

In [ ]:
aprx = arcpy.mp.ArcGISProject('CURRENT')
output_csv = aprx.homeFolder + '\\' + 'query_results.csv'
df.to_csv(output_csv)
arcpy.management.XYTableToPoint(r'U:\STAFF\ColeWhite\sample_projects\REST_API_Query_Example\query_results.csv', 'results7', 'longitude', 'latitude') 